# 05 — Risk Analysis

Comprehensive risk assessment: VaR, CVaR, stress testing, and drawdown analysis.
Includes the 2021 Texas freeze scenario — a critical event for energy trading risk.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import ISODataFetcher
from src.analysis.spreads import SpreadAnalyzer
from src.strategy.mean_reversion import MeanReversionStrategy
from src.strategy.backtest import BacktestEngine
from src.risk.var import RiskMetrics
from src.risk.stress import StressTest
from src.risk.position import PositionSizer

In [ ]:
# Generate backtest results for risk analysis
fetcher = ISODataFetcher(config_path='../config/settings.yaml')
analyzer = SpreadAnalyzer()
engine = BacktestEngine()

end_date = pd.Timestamp.now().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.now() - pd.Timedelta(days=365)).strftime('%Y-%m-%d')

ercot = fetcher.fetch('ERCOT', start_date, end_date)
pjm = fetcher.fetch('PJM', start_date, end_date)
spread_df = analyzer.compute_spread(ercot, pjm)

strategy = MeanReversionStrategy()
signals = strategy.generate_signals(spread_df['spread'])
bt = engine.run(signals)

In [ ]:
# VaR and CVaR
risk = RiskMetrics()
returns = bt['daily_pnl'][1:] / 100_000

report = risk.risk_report(returns, bt['equity_curve'])
print("Risk Report:")
for k, v in report.items():
    if isinstance(v, dict):
        print(f"  {k}:")
        for kk, vv in v.items():
            print(f"    {kk}: {vv:.4f}" if isinstance(vv, float) else f"    {kk}: {vv}")
    else:
        print(f"  {k}: {v:.6f}")

In [ ]:
# Return distribution with VaR markers
fig = go.Figure()
fig.add_trace(go.Histogram(x=returns, nbinsx=50, name='Daily Returns'))
fig.add_vline(x=report['historical_var_95'], line_dash='dash', line_color='orange',
              annotation_text='VaR 95%')
fig.add_vline(x=report['historical_var_99'], line_dash='dash', line_color='red',
              annotation_text='VaR 99%')
fig.add_vline(x=report['cvar_95'], line_dash='dot', line_color='darkred',
              annotation_text='CVaR 95%')
fig.update_layout(title='Return Distribution with VaR/CVaR', height=500)
fig.show()

In [ ]:
# Drawdown analysis
dd_info = risk.max_drawdown(bt['equity_curve'])
peak = bt['equity_curve'].cummax()
drawdown = (bt['equity_curve'] - peak) / peak

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Equity Curve', 'Drawdown'])
fig.add_trace(go.Scatter(y=bt['equity_curve'], name='Equity',
                         fill='tozeroy'), row=1, col=1)
fig.add_trace(go.Scatter(y=drawdown, name='Drawdown',
                         fill='tozeroy', line=dict(color='red')), row=2, col=1)
fig.update_layout(height=600, title_text='Drawdown Analysis')
fig.show()

print(f"Max Drawdown: {dd_info['max_drawdown_pct']:.2f}%")
print(f"Max Duration: {dd_info['max_duration_periods']} periods")

In [ ]:
# Stress Testing
stress = StressTest()
positions = {'ERCOT-PJM': 1}  # long ERCOT-PJM spread

print("Stress Test Results:")
print("=" * 60)
for scenario in stress.SCENARIOS:
    result = stress.run_scenario(scenario, positions)
    print(f"\n{result['description']}")
    print(f"  Estimated P&L: ${result['total_pnl']:,.0f}")
    print(f"  Duration: {result['duration_hours']} hours")

In [ ]:
# Position sizing
sizer = PositionSizer()
metrics = bt['metrics']

sizing = sizer.optimal_size(
    capital=100_000,
    win_rate=metrics['win_rate'],
    avg_win=metrics['avg_win'],
    avg_loss=metrics['avg_loss'],
    max_risk_pct=0.02,
    stop_distance=5.0,  # $5/MWh stop loss
)

print("\nPosition Sizing Recommendations:")
for k, v in sizing.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Risk Management Takeaways

1. **Tail risk is real**: CVaR captures losses beyond VaR — critical for power markets with fat tails
2. **Texas freeze exposure**: A long ERCOT spread would have produced massive P&L (positive or negative depending on direction)
3. **Position sizing**: Kelly criterion provides a theoretical optimum, but half-Kelly is more practical
4. **Drawdown monitoring**: Max drawdown is the strategy's survival metric — if DD exceeds risk tolerance, reduce size